# 09 — UC SQL Function Tool: `predict_hiring_score`

**J&J HRD 2030 Predictive Hiring Demo**

Explores registering the ML prediction as a **Unity Catalog SQL function** that wraps `ai_query()`,
then wiring it into the agent via `UCFunctionToolkit` — replacing the REST-based `predict_hiring_score` tool.

### What this notebook does
1. Creates two UC SQL functions in `bx4.hrd_2030`:
   - `predict_hiring_score(8 ints)` — core scorer, calls `ai_query('hr-predictive-hiring-endpoint', ...)`
   - `predict_hiring_score_by_id(candidate_id)` — looks up scores from Delta, delegates to (1)
2. Tests both functions directly via `spark.sql()`
3. Wraps them as a LangChain tool via `UCFunctionToolkit`
4. Runs a quick agent conversation to confirm tool invocation works end-to-end

### Why UC Functions vs. REST tool?
| | REST tool (`tool_predict_score.py`) | UC SQL Function |
|---|---|---|
| Governance | None | Registered in UC — discoverable, auditable, permissioned |
| Reuse | Agent only | Any notebook, dashboard, SQL query, or agent |
| Auth | Token from env var | Inherited from caller identity |
| Maintenance | Python code | SQL — easier for data teams to own |
| Bulk scoring | One at a time | Native SQL — score entire table in one SELECT |

In [ ]:
%pip install -q databricks-langchain unitycatalog-ai[databricks] mlflow[databricks]

In [ ]:
dbutils.library.restartPython()

## Setup — Widgets & Config

In [ ]:
dbutils.widgets.text("catalog",        "bx4",      "UC Catalog")
dbutils.widgets.text("schema",         "hrd_2030", "UC Schema")
dbutils.widgets.text("endpoint_name",  "hr-predictive-hiring-endpoint", "ML Endpoint")
dbutils.widgets.text("llm_endpoint",   "databricks-gpt-5-4",            "LLM Endpoint")

In [ ]:
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
endpoint_name = dbutils.widgets.get("endpoint_name")
llm_endpoint  = dbutils.widgets.get("llm_endpoint")

print(f"Catalog  : {catalog}")
print(f"Schema   : {schema}")
print(f"Endpoint : {endpoint_name}")

## Step 1 — Create UC SQL Functions

Two functions registered to `bx4.hrd_2030`:
- **`predict_hiring_score`** — takes 8 feature scores, calls `ai_query()` on the ML endpoint
- **`predict_hiring_score_by_id`** — takes a `candidate_id`, looks up scores from Delta, delegates to the first function

In [ ]:
# Core scoring function — wraps ai_query() with the ML endpoint
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.predict_hiring_score(
  education_score          INT  COMMENT 'Education qualification score (0–100)',
  experience_score         INT  COMMENT 'Total experience score (0–100)',
  leadership_score         INT  COMMENT 'Leadership experience score (0–100)',
  certification_score      INT  COMMENT 'Professional certifications score (0–100)',
  skills_match_score       INT  COMMENT 'Skills match to job requirements (0–100)',
  industry_relevance_score INT  COMMENT 'Industry relevance score (0–100)',
  interview_score          INT  COMMENT 'Interview performance score (0–100)',
  culture_fit              INT  COMMENT 'Culture fit score (0–100)'
)
RETURNS STRING
COMMENT 'Call the {endpoint_name} ML model via ai_query to predict hire/no-hire. Returns JSON with prediction (0=no hire, 1=hire).'
RETURN (
  SELECT CAST(
    ai_query(
      '{endpoint_name}',
      named_struct(
        'dataframe_records', array(
          named_struct(
            'education_score',          predict_hiring_score.education_score,
            'experience_score',         predict_hiring_score.experience_score,
            'leadership_score',         predict_hiring_score.leadership_score,
            'certification_score',      predict_hiring_score.certification_score,
            'skills_match_score',       predict_hiring_score.skills_match_score,
            'industry_relevance_score', predict_hiring_score.industry_relevance_score,
            'interview_score',          predict_hiring_score.interview_score,
            'culture_fit',              predict_hiring_score.culture_fit
          )
        )
      )
    ) AS STRING
  )
)
""")

print(f"✓ Created {catalog}.{schema}.predict_hiring_score")

In [ ]:
# Candidate-ID wrapper — looks up from the candidates Delta table, handles missing scores
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.predict_hiring_score_by_id(
  candidate_id STRING COMMENT 'Candidate ID, e.g. C001 or C011'
)
RETURNS STRING
COMMENT 'Predict hire/no-hire for a candidate by their ID. Fetches feature scores from the candidates Delta table and calls the ML model via ai_query. Returns a helpful message for new candidates missing post-interview scores.'
RETURN (
  SELECT
    CASE
      -- Missing base scores — data quality issue
      WHEN education_score IS NULL OR experience_score IS NULL
           OR leadership_score IS NULL OR certification_score IS NULL
           OR industry_relevance_score IS NULL
      THEN CONCAT('Candidate ', predict_hiring_score_by_id.candidate_id,
                  ' is missing required feature scores in the candidates table.')

      -- Missing post-interview scores (new candidates C011–C020)
      WHEN skills_match_score IS NULL OR interview_score IS NULL OR culture_fit IS NULL
      THEN CONCAT(
             'Candidate ', COALESCE(first_name, ''), ' ', COALESCE(last_name, ''),
             ' (', predict_hiring_score_by_id.candidate_id, ') is awaiting interview. ',
             'Please supply: ',
             CASE WHEN skills_match_score IS NULL THEN 'skills_match_score ' ELSE '' END,
             CASE WHEN interview_score    IS NULL THEN 'interview_score '    ELSE '' END,
             CASE WHEN culture_fit        IS NULL THEN 'culture_fit '        ELSE '' END,
             '(each 0–100) after their interview.'
           )

      -- All scores present — call the ML model
      ELSE {catalog}.{schema}.predict_hiring_score(
             CAST(education_score          AS INT),
             CAST(experience_score         AS INT),
             CAST(leadership_score         AS INT),
             CAST(certification_score      AS INT),
             CAST(skills_match_score       AS INT),
             CAST(industry_relevance_score AS INT),
             CAST(interview_score          AS INT),
             CAST(culture_fit             AS INT)
           )
    END
  FROM {catalog}.{schema}.candidates
  WHERE UPPER(candidates.candidate_id) = UPPER(predict_hiring_score_by_id.candidate_id)
  LIMIT 1
)
""")

print(f"✓ Created {catalog}.{schema}.predict_hiring_score_by_id")

## Step 2 — Test Functions Directly via `spark.sql()`

Validate the functions before wiring into the agent.

In [ ]:
# Test 1: Direct ai_query() call matching the user's bulk dataframe_split example
print("Test 1 — bulk ai_query() with dataframe_split format:")
result = spark.sql(f"""
SELECT ai_query(
  '{endpoint_name}',
  request => named_struct(
    'dataframe_split', named_struct(
      'columns', array(
        'education_score', 'experience_score', 'leadership_score',
        'certification_score', 'skills_match_score', 'industry_relevance_score',
        'interview_score', 'culture_fit'
      ),
      'data', array(
        array(85, 65, 55, 75, 60, 55, 65, 70),
        array(75, 85, 80, 90, 85, 90, 82, 85),
        array(60, 70, 45, 65, 58, 45, 62, 65),
        array(85, 95, 92, 95, 95, 95, 90, 92),
        array(85, 90, 85, 95, 92, 95, 88, 88)
      )
    )
  )
) AS bulk_predictions
""")
result.show(truncate=False)

In [ ]:
# Test 2: Core scoring function with explicit feature scores
print("Test 2 — predict_hiring_score() with explicit scores:")
result = spark.sql(f"""
SELECT
  {catalog}.{schema}.predict_hiring_score(85, 90, 85, 95, 92, 95, 88, 88) AS high_scorer,
  {catalog}.{schema}.predict_hiring_score(70, 40, 30, 45, 35, 40, 42, 45) AS low_scorer
""")
result.show(truncate=False)

In [ ]:
# Test 3: Candidate-ID wrapper — existing candidates with full scores
print("Test 3 — predict_hiring_score_by_id() for existing candidates:")
result = spark.sql(f"""
SELECT
  candidate_id,
  first_name,
  last_name,
  {catalog}.{schema}.predict_hiring_score_by_id(candidate_id) AS ml_prediction,
  hired                                                        AS actual_label
FROM {catalog}.{schema}.candidates
WHERE candidate_id BETWEEN 'C001' AND 'C010'
ORDER BY candidate_id
""")
result.show(truncate=False)

In [ ]:
# Test 4: New candidate (C011) — should return helpful message about missing scores
print("Test 4 — new candidate missing post-interview scores:")
result = spark.sql(f"""
SELECT {catalog}.{schema}.predict_hiring_score_by_id('C011') AS result
""")
result.show(truncate=False)

## Step 3 — Wire as a LangChain Tool via `UCFunctionToolkit`

Use `DatabricksFunctionClient` + `UCFunctionToolkit` to wrap the UC function as a LangChain `@tool`.
This is the same pattern used in the weatherwise reference agent.

In [ ]:
from databricks_langchain import DatabricksFunctionClient, UCFunctionToolkit

# DatabricksFunctionClient auto-authenticates using the notebook's Databricks context
uc_client = DatabricksFunctionClient()

# Expose predict_hiring_score_by_id as a LangChain tool
# The docstring from the CREATE FUNCTION COMMENT becomes the tool description
uc_toolkit = UCFunctionToolkit(
    function_names=[f"{catalog}.{schema}.predict_hiring_score_by_id"],
    client=uc_client,
)

uc_tools = uc_toolkit.tools

print(f"✓ UC tool registered: {len(uc_tools)} tool(s)")
for t in uc_tools:
    print(f"  • {t.name}")
    print(f"    {t.description[:120]}...")

In [ ]:
# Invoke the UC tool directly (as the agent would) — confirms round-trip works
predict_tool = uc_tools[0]

print("Invoking UC tool for C004 (David Kim)...")
result_c004 = predict_tool.invoke({"candidate_id": "C004"})
print(f"C004 result: {result_c004}")

print("\nInvoking UC tool for C006 (Robert Johnson — low scorer)...")
result_c006 = predict_tool.invoke({"candidate_id": "C006"})
print(f"C006 result: {result_c006}")

print("\nInvoking UC tool for C011 (new candidate — missing scores)...")
result_c011 = predict_tool.invoke({"candidate_id": "C011"})
print(f"C011 result: {result_c011}")

## Step 4 — Quick Agent Conversation

Swap out the REST-based `predict_hiring_score` tool for the UC function tool and
run a quick conversation to confirm the agent can invoke it correctly.

In [ ]:
import os, uuid
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from databricks_langchain import ChatDatabricks

# Bring in the other agent tools (Genie, resume search, email)
# and replace the REST predict tool with the UC function tool
import sys
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root  = "/Workspace" + notebook_path.rsplit("/notebooks", 1)[0]
sys.path.insert(0, f"{project_root}/agent_src")

from tools.tool_query_genie   import query_genie
from tools.tool_search_resume import search_resumes
from tools.tool_send_email    import send_email

# Set env vars needed by Genie and VS tools
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
os.environ.setdefault("DATABRICKS_HOST",  ctx.apiUrl().get())
os.environ.setdefault("DATABRICKS_TOKEN", ctx.apiToken().get())

# Mix: REST tools + UC function tool for prediction
tools_with_uc_predict = [query_genie, search_resumes, uc_tools[0], send_email]

print(f"Agent tools ({len(tools_with_uc_predict)}):")
for t in tools_with_uc_predict:
    print(f"  • {t.name}")

In [ ]:
# Simple tool-calling loop — same pattern as HireRightAgent.predict()
llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0.1, max_tokens=1024)
llm_with_tools = llm.bind_tools(tools_with_uc_predict)

messages = [
    SystemMessage(content="You are a helpful hiring assistant. Use available tools to answer questions."),
    HumanMessage(content="Run the ML model prediction for David Kim (C004) and tell me the result."),
]

print("Running agent conversation with UC function tool...\n")

for step in range(5):
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        print("Final answer:")
        print(response.content)
        break

    for tc in response.tool_calls:
        print(f"  → Tool call: {tc['name']}({tc['args']})")
        result = "Tool not found"
        for t in tools_with_uc_predict:
            if t.name == tc["name"]:
                result = t.invoke(tc["args"])
                break
        print(f"    Result: {str(result)[:200]}")
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

## Summary

| Resource | Detail |
|---|---|
| UC function (core) | `bx4.hrd_2030.predict_hiring_score(8 ints)` |
| UC function (wrapper) | `bx4.hrd_2030.predict_hiring_score_by_id(candidate_id)` |
| ML endpoint called | `hr-predictive-hiring-endpoint` via `ai_query()` |
| Tool framework | `UCFunctionToolkit` from `databricks-langchain` |
| SQL scripts | `scripts/01_create_uc_predict_functions.sql` |

### Advantages of the UC Function approach
- **Governed**: registered in Unity Catalog — discoverable in Catalog Explorer, auditable, permissionable
- **Reusable**: callable from any SQL query, dashboard, notebook, or agent without Python code
- **Bulk scoring**: `SELECT predict_hiring_score_by_id(candidate_id) FROM candidates` scores everyone in one query
- **No auth plumbing**: `ai_query` inherits the caller's identity — no token management

### Next step
If testing passes, replace `tool_predict_score.py` in `agent_src/tools/` with the
`UCFunctionToolkit`-based tool and re-register the agent.